[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/EMQA/blob/main/EMQA_cross_market_analysis/EMQA_cross_market_analysis.ipynb)

# EMQA_cross_market_analysis

Cross-market analysis of Romanian (OPCOM) vs German (EPEX) day-ahead
electricity prices, 2020–2026. Four panels: time series overlay, scatter
with regression, rolling correlation, and lead-lag cross-correlation.

**Output:** `ml_cross_market_analysis.pdf`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'none',
    'axes.facecolor': 'none',
    'savefig.facecolor': 'none',
    'savefig.transparent': True,
    'axes.grid': False,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'figure.figsize': (12, 6),
})

COLORS = {
    'blue': '#1A3A6E', 'red': '#CD0000', 'green': '#2E7D32',
    'orange': '#E67E22', 'purple': '#8E44AD', 'gray': '#808080',
    'cyan': '#00BCD4', 'amber': '#B5853F'
}

def save_fig(fig, name):
    fig.savefig(name, bbox_inches='tight', transparent=True, dpi=300)
    print(f'Saved: {name}')

In [ ]:
url = 'https://raw.githubusercontent.com/QuantLet/EMQA/main/EMQA_feature_importance/ro_de_prices_full.csv'
ro = pd.read_csv(url, parse_dates=['date'], index_col='date')
print(f'Loaded {len(ro)} observations, {ro.index[0].date()} to {ro.index[-1].date()}')

rho = ro['ro_price'].corr(ro['de_price'])
print(f'Overall correlation RO-DE: {rho:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

leg_kw = dict(fontsize=9, loc='upper center', bbox_to_anchor=(0.5, -0.12),
              frameon=False, ncol=3)

# --- Panel 1: Time series overlay ---
ax = axes[0, 0]
ax.plot(ro.index, ro['ro_price'], color=COLORS['blue'], alpha=0.8, lw=0.8, label='Romania (OPCOM)')
ax.plot(ro.index, ro['de_price'], color=COLORS['red'], alpha=0.8, lw=0.8, label='Germany (EPEX)')
ax.set_ylabel('Price (\u20ac/MWh)', fontsize=10)
ax.set_title('Day-Ahead Prices: RO vs DE', fontsize=12, fontweight='bold')
ax.legend(**leg_kw)

# --- Panel 2: Scatter with regression ---
ax = axes[0, 1]
ax.scatter(ro['de_price'], ro['ro_price'], alpha=0.25, s=8, color=COLORS['blue'], edgecolors='none')
slope, intercept, r_val, p_val, se = stats.linregress(ro['de_price'], ro['ro_price'])
x_line = np.linspace(ro['de_price'].min(), ro['de_price'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, color=COLORS['red'], lw=2,
        label=f'OLS: y = {slope:.2f}x + {intercept:.1f}')
ax.annotate(f'\u03c1 = {rho:.2f}', xy=(0.05, 0.92), xycoords='axes fraction',
            fontsize=14, fontweight='bold', color=COLORS['red'])
ax.set_xlabel('DE Price (\u20ac/MWh)', fontsize=10)
ax.set_ylabel('RO Price (\u20ac/MWh)', fontsize=10)
ax.set_title('Scatter: RO vs DE Prices', fontsize=12, fontweight='bold')
ax.legend(**leg_kw)

# --- Panel 3: Rolling 30-day correlation ---
ax = axes[1, 0]
rolling_corr = ro['ro_price'].rolling(30).corr(ro['de_price'])
ax.plot(ro.index, rolling_corr, color=COLORS['green'], lw=1)
ax.axhline(rho, color=COLORS['gray'], ls='--', lw=1, label=f'Overall \u03c1 = {rho:.2f}')
ax.set_ylabel('Correlation', fontsize=10)
ax.set_title('Rolling 30-Day Correlation', fontsize=12, fontweight='bold')
ax.set_ylim(-0.2, 1.05)
ax.legend(**leg_kw)

# --- Panel 4: Lead-lag cross-correlation ---
ax = axes[1, 1]
max_lag = 14
lags = range(-max_lag, max_lag + 1)
ccf = [ro['de_price'].shift(k).corr(ro['ro_price']) for k in lags]
colors = [COLORS['red'] if abs(c) == max(abs(v) for v in ccf) else COLORS['blue'] for c in ccf]
ax.bar(lags, ccf, color=colors, alpha=0.7, width=0.8)
ax.axhline(0, color='black', lw=0.5)
peak_lag = list(lags)[np.argmax(ccf)]
ax.annotate(f'Peak at lag {peak_lag}', xy=(peak_lag, max(ccf)),
            xytext=(peak_lag + 3, max(ccf) - 0.05),
            arrowprops=dict(arrowstyle='->', color=COLORS['red']),
            fontsize=10, color=COLORS['red'])
ax.set_xlabel('Lag (days, positive = DE leads RO)', fontsize=10)
ax.set_ylabel('Cross-Correlation', fontsize=10)
ax.set_title('Lead-Lag Analysis: DE \u2192 RO', fontsize=12, fontweight='bold')

fig.suptitle('Cross-Market Analysis: Romania vs Germany',
             fontsize=14, fontweight='bold', y=1.01)
fig.subplots_adjust(hspace=0.45, wspace=0.3)
save_fig(fig, 'ml_cross_market_analysis.pdf')
plt.show()

print(f'\n=== Summary ===')
print(f'  Overall correlation: {rho:.4f}')
print(f'  OLS slope: {slope:.3f}, intercept: {intercept:.1f}')
print(f'  Peak cross-correlation at lag {peak_lag} days')